# 04 — Travel times & accessibility (Phase 4)

Every hex gets travel times to every other hex (**walk** with Tobler's
hiking function on the DEM grades, **bike** with an up/down speed
adjustment, **transit**, **car** free-flow as a reference) and
accessibility measures on top: cumulative opportunities, gravity access,
nearest facilities and 2SFCA for capacity-constrained services.

> **Transit is approximate.** Almaty publishes no GTFS feed, so the
> transit network is rebuilt from OSM route relations with *assumed*
> commercial speeds and headways (`access.transit` in the config:
> expected wait = half the headway, paid at every boarding). Treat every
> transit number as *approximate transit access*, not timetable truth.

By default this notebook runs on the committed sample (a 2 km core
window — walking distances only); set `ALMATY_CONFIG=config/config.yaml`
to reproduce the full-city figures.

In [ ]:
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from almaty_access.access.build import REPORT_FILE, build_measures, build_times
from almaty_access.config import load_config
from almaty_access.prepare.transit import LEGS_FILE, ROUTES_FILE, STOPS_FILE, prepare_transit

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
CONFIG_PATH = Path(os.environ.get("ALMATY_CONFIG", REPO / "config" / "config.sample.yaml"))
config = load_config(CONFIG_PATH)
prepare_transit(config)  # all cached
build_times(config)
build_measures(config)

P = config.paths.processed
access8 = pd.read_parquet(P / "access_r8.parquet")
access9 = pd.read_parquet(P / "access_r9.parquet")
grid8 = gpd.read_parquet(P / "grid_r8.parquet")
grid9 = gpd.read_parquet(P / "grid_r9.parquet")
report = json.loads((P / REPORT_FILE).read_text("utf-8"))
stops = gpd.read_parquet(config.paths.interim / STOPS_FILE)
routes = pd.read_parquet(config.paths.interim / ROUTES_FILE)
legs = pd.read_parquet(config.paths.interim / LEGS_FILE)

net = report["transit_network"]
print(f"config: {CONFIG_PATH.name}")
print(
    f"transit: kept {net['routes_kept']} routes {net['routes_kept_by_mode']} "
    f"with {net['stops_kept']} stops / {net['legs']} legs "
    f"({net['relations_without_stops']} relations had no usable stops, "
    f"{net['routes_outside_area']} fell outside the area)"
)
for name, stats in report["matrices"].items():
    extra = f" ({stats['skipped_reason']})" if "skipped_reason" in stats else ""
    print(f"{name}: {stats['pairs']:,} pairs{extra}")

## The approximate transit network

Stops and legs that survived clipping to the study area, coloured by
mode. Legs are drawn as straight stop-to-stop lines — exactly how the
model sees them (street-path length is approximated as straight-line
distance × circuity, so what you see is what the travel times use).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
grid8.boundary.plot(ax=ax, linewidth=0.2, color="#bbbbbb", zorder=1)
stop_xy = stops.set_index("stop_id").geometry
colors = {"bus": "#1f77b4", "trolleybus": "#2ca02c", "subway": "#d62728"}
mode_of_route = routes.set_index("route_id")["mode"]
for mode, group in legs.groupby(mode_of_route.reindex(legs["route_id"]).to_numpy()):
    for row in group.itertuples(index=False):
        a, b = stop_xy.get(row.from_stop), stop_xy.get(row.to_stop)
        if a is None or b is None:
            continue
        ax.plot([a.x, b.x], [a.y, b.y], color=colors[mode], linewidth=0.8, alpha=0.6, zorder=2)
if len(stops):
    stops.plot(ax=ax, markersize=4, color="#333333", zorder=3)
handles = [plt.Line2D([0], [0], color=c, label=m) for m, c in colors.items()]
ax.legend(handles=handles, loc="upper right")
ax.set_title(f"Approximate transit network: {len(routes)} routes, {len(stops)} stops")
ax.set_axis_off()
plt.show()

print(routes.groupby("mode")["headway_min"].describe().round(1))

## Jobs reachable: walk vs transit

Cumulative access `acc_jobs_{mode}_{T}` counts the Phase 3 jobs within
*T* minutes. Walking is mapped at the fine r9 resolution; transit at the
r8 analysis resolution. The percentage versions (`accpct_`) divide by
all city jobs at that resolution, which is what makes modes comparable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

walk9 = grid9.merge(access9[["h3", "acc_jobs_walk_30"]], on="h3")
walk9.plot(column="acc_jobs_walk_30", ax=axes[0], legend=True, cmap="viridis", linewidth=0)
axes[0].set_title("Jobs within 30 min walk (r9)")

transit8 = grid8.merge(access8[["h3", "accpct_jobs_transit_45"]], on="h3")
transit8.plot(column="accpct_jobs_transit_45", ax=axes[1], legend=True, cmap="viridis", linewidth=0)
axes[1].set_title("% of city jobs within 45 min transit (r8, approximate)")

gap = grid8.merge(access8[["h3", "accpct_jobs_transit_45", "accpct_jobs_car_45"]], on="h3")
gap["ratio"] = np.where(
    gap["accpct_jobs_car_45"] > 0,
    gap["accpct_jobs_transit_45"] / gap["accpct_jobs_car_45"],
    np.nan,
)
gap.plot(column="ratio", ax=axes[2], legend=True, cmap="RdYlGn", vmin=0, vmax=1, linewidth=0)
axes[2].set_title("Transit / car access ratio at 45 min (Phase 5 preview)")

for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

city8 = access8.loc[access8["in_city"]]
for mode in ("walk", "bike", "transit", "car"):
    print(
        f"{mode:8s} median jobs 45 min: {city8[f'acc_jobs_{mode}_45'].median():12,.0f}   "
        f"share of hexes reaching any job in 30: "
        f"{(city8[f'acc_jobs_{mode}_30'] > 0).mean():.0%}"
    )

## Services on foot: nearest facility and 2SFCA

`near_*_walk_min` answers "how far is the closest school / clinic /
park / metro?". The **two-step floating catchment area** goes further
for capacity-constrained services: step 1 gives every facility hex a
supply-to-demand ratio (its facilities divided by the *population that
can reach it* within the catchment), step 2 sums those ratios over the
facilities each hex can reach. A clinic shared with a dense bedroom
district counts for less than one all to yourself — that is the
competition effect a plain count misses. Values are facilities per
1,000 residents within the catchment.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
near9 = grid9.merge(access9[["h3", "near_clinic_walk_min"]], on="h3")
near9.plot(
    column="near_clinic_walk_min",
    ax=axes[0],
    legend=True,
    cmap="magma_r",
    linewidth=0,
    missing_kwds={"color": "#dddddd"},
)
axes[0].set_title("Minutes to the nearest clinic, walking (r9)")

sfca9 = grid9.merge(access9[["h3", "sfca_clinics_walk"]], on="h3")
sfca9.plot(column="sfca_clinics_walk", ax=axes[1], legend=True, cmap="viridis", linewidth=0)
axes[1].set_title(
    f"2SFCA clinics per 1,000 residents ({config.access.sfca_catchment_min} min walk)"
)
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

near_cols = [c for c in access9.columns if c.startswith("near_")]
print(access9.loc[access9["in_city"], near_cols].describe().loc[["25%", "50%", "75%"]].round(1))

## Spot checks against a routing app (README's done-when)

Ten seeded origin–destination pairs. Open each pair in 2GIS / Yandex
Maps and compare: walking should land within a few minutes of the app;
car (free-flow) should be *faster or equal to* an off-peak app estimate
(we model no congestion); transit should be in the right ballpark given
the assumed headways. Record your verdicts in DECISIONS.md.

In [ ]:
rows = []
for check in report["spot_checks"]:
    o_lat, o_lon = check["origin_latlng"]
    d_lat, d_lon = check["dest_latlng"]
    rows.append(
        {
            "origin": f"{o_lat:.4f},{o_lon:.4f}",
            "dest": f"{d_lat:.4f},{d_lon:.4f}",
            **{f"{m}_min": v for m, v in check["minutes"].items()},
        }
    )
spot = pd.DataFrame(rows)
print(spot.to_string(index=False))
print(
    "\nPaste 'lat,lon' into 2GIS/Yandex as A and B. None = beyond the "
    f"{int(report['params']['cutoff_min'])}-minute cutoff for that mode."
)

## Gravity sensitivity and the verdict

The gravity measure needs a decay parameter; we derive it from a
half-life (jobs at 20 min count half as much as jobs next door). The
report's sensitivity block recomputes the surface at 10 and 30 min
half-lives and reports the Spearman rank correlation with the central
one: if the *ranking* of hexes barely moves, no conclusion of Phase 5
hangs on the exact β.

In [ ]:
import itertools

failures = []


def check(name: str, condition: bool) -> None:
    print(("PASS " if condition else "FAIL ") + name)
    if not condition:
        failures.append(name)


thresholds = list(config.travel.thresholds_min)
modes8 = ("walk", "bike", "transit", "car")

check("report labels transit as approximate", report["approximate_transit"] is True)
check(
    "all five matrices exist",
    all((P / f"tt_{m}_r8.parquet").exists() for m in modes8)
    and (P / "tt_walk_r9.parquet").exists(),
)

for mode in modes8:
    monotone = all(
        (access8[f"acc_jobs_{mode}_{a}"] <= access8[f"acc_jobs_{mode}_{b}"] + 1e-6).all()
        for a, b in itertools.pairwise(thresholds)
    )
    check(f"{mode}: cumulative access monotone over thresholds", monotone)

check(
    "percentages within [0, 100]",
    all(
        access8[f"accpct_jobs_{m}_{t}"].between(-1e-9, 100 + 1e-9).all()
        for m in modes8
        for t in thresholds
    ),
)

walk_tt = pd.read_parquet(P / "tt_walk_r8.parquet").set_index(["origin_h3", "dest_h3"])["minutes"]
transit_tt = pd.read_parquet(P / "tt_transit_r8.parquet")
if len(transit_tt):
    joined = pd.concat(
        [walk_tt, transit_tt.set_index(["origin_h3", "dest_h3"])["minutes"]],
        axis=1,
        keys=["walk", "transit"],
    ).dropna()
    check(
        "transit never slower than walking (same pairs)",
        bool((joined["transit"] <= joined["walk"] + 0.05).all()),
    )

check(
    "gravity access never exceeds the 60-min cumulative",
    all(
        (access8[f"acc_jobs_{m}_grav"] <= access8[f"acc_jobs_{m}_60"] + 1e-6).all() for m in modes8
    ),
)
check(
    "nearest-facility minutes are non-negative",
    bool((access9.filter(like="near_").min() >= 0).all()),
)
check(
    "2SFCA values are finite and non-negative",
    bool(
        np.isfinite(access9.filter(like="sfca_")).all().all()
        and (access9.filter(like="sfca_") >= 0).all().all()
    ),
)
check("10 spot-check pairs recorded", len(report["spot_checks"]) == 10)

sens = report["gravity_sensitivity"]["spearman_vs"]
print(f"gravity rank stability: {sens}")
check("gravity ranking robust to the half-life (rho >= 0.9)", all(v >= 0.9 for v in sens.values()))

city_share = report["coverage"][f"r{config.h3.analysis_resolution}"]["walk"][
    "share_reaching_jobs_30"
]
check("most city hexes reach jobs within a 30-min walk", city_share > 0.5)

print()
if failures:
    raise AssertionError(f"verdict failed: {failures}")
print("All checks passed. Accessibility surfaces are ready for Phase 5 equity.")